# Archive VinDr-Mammo Dataset with Files

This notebook creates a complete archive of the stratified dataset including:
- All DICOM image files
- Original folder structure preserved
- Split metadata and indices

**Output:** `vindr_mammo_dataset.zip` with complete dataset

In [ ]:
import os
import json
import zipfile
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

## Configuration

In [ ]:
# Paths - UPDATE THESE
VINDR_CSV = "path/to/vindr_mammo.csv"  # Update this path
VINDR_IMAGES_ROOT = "path/to/vindr_images"  # Update this path
OUTPUT_ZIP = "vindr_mammo_dataset.zip"

# Split parameters
TRAIN_RATIO = 0.8
RANDOM_STATE = 42
BENIGN_BIRADS = [1, 2, 3]
MALIGNANT_BIRADS = [5, 6]

## Step 1: Load Dataset and Create Split

In [ ]:
# Import utilities
import sys
from pathlib import Path as PathLib

project_root = PathLib.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from breast_cancer_detection.src.datasets import (
    VinDRMammoBinaryDataset,
    create_breast_level_splits,
    map_birads_to_binary
)
from breast_cancer_detection.src.preprocessing import MammographyPreprocessor

In [ ]:
# Create preprocessor (needed for dataset initialization)
preprocessor = MammographyPreprocessor(
    target_size=(720, 480),
    aspect_ratio=1.5
)

# Load dataset
print("Loading VinDr-Mammo dataset...")
full_dataset = VinDRMammoBinaryDataset(
    images_root=VINDR_IMAGES_ROOT,
    csv_file=VINDR_CSV,
    preprocessor=preprocessor,
    transform=None,
    benign_birads=BENIGN_BIRADS,
    malignant_birads=MALIGNANT_BIRADS
)

print(f"Total samples: {len(full_dataset)}")

In [ ]:
# Create stratified split
print("Creating stratified breast-level split...")
train_subset, val_subset = create_breast_level_splits(
    dataset=full_dataset,
    train_ratio=TRAIN_RATIO,
    random_state=RANDOM_STATE,
    stratify=True
)

train_indices = train_subset.indices
val_indices = val_subset.indices

print(f"\nTrain samples: {len(train_indices)}")
print(f"Validation samples: {len(val_indices)}")

## Step 2: Prepare Archive Structure

In [ ]:
# Create temporary directory structure
temp_root = Path("temp_dataset_archive")
temp_root.mkdir(exist_ok=True)

# Create subdirectories
train_dir = temp_root / "train"
val_dir = temp_root / "val"
metadata_dir = temp_root / "metadata"

train_dir.mkdir(exist_ok=True)
val_dir.mkdir(exist_ok=True)
metadata_dir.mkdir(exist_ok=True)

print(f"Created directory structure at: {temp_root}")

## Step 3: Copy DICOM Files with Folder Structure

In [ ]:
def copy_dicom_files(indices, target_dir, dataset, images_root):
    """
    Copy DICOM files preserving folder structure.
    
    Structure:
    target_dir/
      study_id_1/
        image_id_1.dicom
        image_id_2.dicom
      study_id_2/
        image_id_3.dicom
    """
    images_root = Path(images_root)
    target_dir = Path(target_dir)
    
    copied_files = 0
    missing_files = []
    
    for idx in tqdm(indices, desc=f"Copying to {target_dir.name}"):
        study_id, image_id, laterality, view_position, label = dataset.samples[idx]
        
        # Source path
        src_path = images_root / str(study_id) / f"{image_id}.dicom"
        
        if not src_path.exists():
            missing_files.append(str(src_path))
            continue
        
        # Destination path (preserve folder structure)
        dst_study_dir = target_dir / str(study_id)
        dst_study_dir.mkdir(exist_ok=True)
        
        dst_path = dst_study_dir / f"{image_id}.dicom"
        
        # Copy file
        shutil.copy2(src_path, dst_path)
        copied_files += 1
    
    return copied_files, missing_files

In [ ]:
# Copy training files
print("\nCopying training files...")
train_copied, train_missing = copy_dicom_files(
    train_indices,
    train_dir,
    full_dataset,
    VINDR_IMAGES_ROOT
)

print(f"✓ Copied {train_copied} training files")
if train_missing:
    print(f"⚠ Missing {len(train_missing)} training files")

In [ ]:
# Copy validation files
print("\nCopying validation files...")
val_copied, val_missing = copy_dicom_files(
    val_indices,
    val_dir,
    full_dataset,
    VINDR_IMAGES_ROOT
)

print(f"✓ Copied {val_copied} validation files")
if val_missing:
    print(f"⚠ Missing {len(val_missing)} validation files")

## Step 4: Save Metadata

In [ ]:
# Create metadata
split_metadata = {
    "created_at": datetime.now().isoformat(),
    "train_ratio": TRAIN_RATIO,
    "random_state": RANDOM_STATE,
    "benign_birads": BENIGN_BIRADS,
    "malignant_birads": MALIGNANT_BIRADS,
    "total_samples": len(full_dataset),
    "train_samples": len(train_indices),
    "val_samples": len(val_indices),
    "train_files_copied": train_copied,
    "val_files_copied": val_copied,
    "split_type": "breast_level_stratified",
    "folder_structure": "train/{study_id}/{image_id}.dicom"
}

# Save metadata
with open(metadata_dir / "split_metadata.json", 'w') as f:
    json.dump(split_metadata, f, indent=2)

# Save indices
np.save(metadata_dir / "train_indices.npy", np.array(train_indices))
np.save(metadata_dir / "val_indices.npy", np.array(val_indices))

# Save sample info
train_samples_df = pd.DataFrame(
    [full_dataset.samples[i] for i in train_indices],
    columns=["study_id", "image_id", "laterality", "view_position", "label"]
)
train_samples_df.to_csv(metadata_dir / "train_samples.csv", index=False)

val_samples_df = pd.DataFrame(
    [full_dataset.samples[i] for i in val_indices],
    columns=["study_id", "image_id", "laterality", "view_position", "label"]
)
val_samples_df.to_csv(metadata_dir / "val_samples.csv", index=False)

print("\n✓ Saved metadata files")

In [ ]:
# Create README
readme_content = f"""VinDr-Mammo Stratified Dataset Archive
=======================================

Created: {split_metadata['created_at']}
Total Samples: {split_metadata['total_samples']}
Train Samples: {split_metadata['train_samples']}
Val Samples: {split_metadata['val_samples']}

Directory Structure:
--------------------
train/
  {'{study_id}/'}
    {'{image_id}'}.dicom
val/
  {'{study_id}/'}
    {'{image_id}'}.dicom
metadata/
  split_metadata.json      - Complete split information
  train_indices.npy        - Training indices
  val_indices.npy          - Validation indices
  train_samples.csv        - Training sample details
  val_samples.csv          - Validation sample details

Split Details:
--------------
- Split Type: Breast-level stratified
- Random Seed: {RANDOM_STATE}
- Train Ratio: {TRAIN_RATIO}
- Benign BI-RADS: {BENIGN_BIRADS}
- Malignant BI-RADS: {MALIGNANT_BIRADS}

Usage:
------
# Load training data
train_df = pd.read_csv('metadata/train_samples.csv')

# Access DICOM files
for idx, row in train_df.iterrows():
    dicom_path = f"train/{'{row.study_id}'}/{'{row.image_id}'}.dicom"
    # Process dicom_path...

Notes:
------
- This is a FIXED split for reproducibility
- CC and MLO views from the same breast are kept together
- Folder structure is preserved from original dataset
"""

with open(temp_root / "README.txt", 'w') as f:
    f.write(readme_content)

print("✓ Created README")

## Step 5: Create ZIP Archive

In [ ]:
print(f"\nCreating ZIP archive: {OUTPUT_ZIP}")
print("This may take several minutes depending on dataset size...\n")

# Get all files to zip
all_files = list(temp_root.rglob("*"))
all_files = [f for f in all_files if f.is_file()]

print(f"Total files to archive: {len(all_files)}")

# Create archive with progress bar
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in tqdm(all_files, desc="Archiving"):
        # Get relative path from temp_root
        arcname = file_path.relative_to(temp_root)
        zipf.write(file_path, arcname=arcname)

archive_size_mb = os.path.getsize(OUTPUT_ZIP) / (1024 * 1024)
print(f"\n✓ Archive created: {OUTPUT_ZIP}")
print(f"  Size: {archive_size_mb:.2f} MB")

## Step 6: Clean Up and Verify

In [ ]:
# Clean up temporary directory
print("\nCleaning up temporary files...")
shutil.rmtree(temp_root)
print("✓ Temporary files removed")

In [ ]:
# Verify archive structure
print(f"\nVerifying archive: {OUTPUT_ZIP}")
print("="*60)

with zipfile.ZipFile(OUTPUT_ZIP, 'r') as zipf:
    # Count files by directory
    train_files = [f for f in zipf.namelist() if f.startswith('train/') and f.endswith('.dicom')]
    val_files = [f for f in zipf.namelist() if f.startswith('val/') and f.endswith('.dicom')]
    metadata_files = [f for f in zipf.namelist() if f.startswith('metadata/')]
    
    print(f"Training DICOMs: {len(train_files)}")
    print(f"Validation DICOMs: {len(val_files)}")
    print(f"Metadata files: {len(metadata_files)}")
    print(f"\nTotal files in archive: {len(zipf.namelist())}")
    
    print("\nSample structure:")
    for i, name in enumerate(sorted(zipf.namelist())[:10]):
        print(f"  {name}")
    if len(zipf.namelist()) > 10:
        print(f"  ... and {len(zipf.namelist()) - 10} more files")

print("\n" + "="*60)
print("✓ Archive created successfully!")
print(f"\nYou can now share or backup: {OUTPUT_ZIP}")